# 1D CNN on Raw RV Data — OOF Evaluation

**Protocol identically mirrors `lightgbm.ipynb` (5-fold StratifiedKFold × 5 reps, seeds 42–46) for honest head-to-head comparison.**

- **Architecture**: StrippedCNN from `cnn_stripped.ipynb` — 3 Conv1d layers (4→64→64→32), masked avg+max global pool, compact classifier head. ~30K params.
- **Input**: 4 raw per-observation channels — `rv_centered`, `rv_err`, `RHKp`, `Halpha`. **No positional encoding** (the 16 sinusoidal BJD channels are stripped).
- **Training per fold**: 60 fixed epochs (no early stopping — pure OOF), Adam(lr=1e-3, wd=5e-3), cosine LR with 5-epoch warmup, `pos_weight=√(n_neg/n_pos)`, grad clip=1.0.
- **Hyperparameters are FIXED** — no CV search, no per-seed tuning.
- **Comparison**: LightGBM OOF PR-AUC = 0.6150 (39 V4 features). This CNN has access to the raw observation sequence but no hand-engineered aggregate statistics.

**Total fits: 25** (5 folds × 5 reps). Each fit trains a fresh CNN from scratch on ~1620 train stars and predicts ~406 held-out stars.

## 1. Imports, configuration, data load

In [1]:
import os
os.environ['PYTHONHASHSEED'] = '0'
import warnings
warnings.filterwarnings('ignore')

import json as _json
import math
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)

seed = 42
np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==== CONFIG (fixed hyperparameters — no CV search) =========================
FEATURE_COLS   = ['rv_centered', 'rv_err', 'RHKp', 'Halpha']  # 4 raw channels, no positional encoding
N_INPUT        = len(FEATURE_COLS)                              # 4
MAX_SEQ_LEN    = 100           # truncate long-cadence stars (matches cnn_stripped)
N_EPOCHS       = 60            # fixed epochs per fold (no early stopping)
BATCH_SIZE     = 32
LR             = 1e-3
WEIGHT_DECAY   = 5e-3
DROPOUT        = 0.3
HIDDEN         = 64
TAIL           = 32
HEAD_HIDDEN    = 32
WARMUP_EPOCHS  = 5
GRAD_CLIP      = 1.0

# OOF protocol — identical to lightgbm.ipynb
N_REPS   = 5
N_FOLDS  = 5

# Paths
OBS_PKL = '/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'

observations = pd.read_pickle(OBS_PKL)
print(f'Observations: {len(observations)} rows, {observations["star_name"].nunique()} stars')
print(f'Device: {device}')
print(f'Features: {FEATURE_COLS}')
print(f'Config: epochs={N_EPOCHS}, batch={BATCH_SIZE}, lr={LR}, wd={WEIGHT_DECAY}, dropout={DROPOUT}')

Observations: 220318 rows, 2026 stars
Device: cuda
Features: ['rv_centered', 'rv_err', 'RHKp', 'Halpha']
Config: epochs=60, batch=32, lr=0.001, wd=0.005, dropout=0.3


## 2. Star vocabulary + labels

Reproduce the alphabetical `groupby('star_name', sort=True)` ordering from `lightgbm.ipynb` / `split.py` so star indices are consistent across all OOF notebooks.

In [2]:
grouped  = observations.groupby('star_name', sort=True)
stars    = list(grouped.groups.keys())
labels   = np.array([int(grouped.get_group(s)['has_exoplanets'].iloc[0]) for s in stars],
                    dtype=int)
n_stars  = len(stars)
print(f'n_stars = {n_stars}, positives = {int(labels.sum())}, negatives = {int((labels==0).sum())}')
print(f'Class ratio 1:{n_stars / max(labels.sum(), 1):.1f}')

n_stars = 2026, positives = 430, negatives = 1596
Class ratio 1:4.7


## 3. Build per-star raw observation sequences

Each star → a `(n_obs_star, 4)` float32 array of `[rv_centered, rv_err, RHKp, Halpha]`, sorted by BJD. No positional encoding, no feature engineering — purely the raw measurement stream.

In [3]:
star_groups = {s: grouped.get_group(s).sort_values('bjd') for s in stars}

def build_sequence(star):
    g = star_groups[star]
    return g[FEATURE_COLS].values.astype(np.float32)

print(f'Building sequences for {n_stars} stars...')
star_seqs = {s: build_sequence(s) for s in stars}
seq_lens  = [len(star_seqs[s]) for s in stars]
print(f'Done. Sequence lengths: min={min(seq_lens)}, max={max(seq_lens)}, median={int(np.median(seq_lens))}')
print(f'First star: {stars[0]}, shape={star_seqs[stars[0]].shape}')

Building sequences for 2026 stars...
Done. Sequence lengths: min=18, max=11469, median=41
First star: 0748-01711-1, shape=(21, 4)


## 4. Model architecture

**StrippedCNN** — the same architecture as `cnn_stripped.ipynb`. Three `same`-padded Conv1d blocks (kernels 5,5,3) each with BatchNorm+GELU+Dropout, followed by a masked concat of global-average and global-max pooling, and a compact 2-layer classifier head.

- **Conv1**: 4 → 64 (kernel 5)
- **Conv2**: 64 → 64 (kernel 5)
- **Conv3**: 64 → 32 (kernel 3)
- **Pool**: masked global avg + max → 64 channels
- **Head**: Linear(64→32) → GELU → Dropout → Linear(32→1)

**~30K parameters** — deliberately small for a ~2000-star dataset.

In [4]:
class MaskedConv1dBlock(nn.Module):
    """Conv1d('same') + BatchNorm + GELU + Dropout."""
    def __init__(self, in_ch, out_ch, kernel_size, dropout=0.3):
        super().__init__()
        pad = kernel_size // 2  # 'same' padding (kernel is odd)
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=pad)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(self.bn(self.conv(x))))


class MaskedAvgMaxPool(nn.Module):
    """Concatenates masked global mean and masked global max pooling.

    Input: (B, C, T), mask (B, T) with True = real obs.
    Output: (B, 2C).
    """
    def forward(self, x, mask):
        m = mask.unsqueeze(1).type_as(x)  # (B, 1, T)
        denom = m.sum(dim=2).clamp_min(1.0)
        avg = (x * m).sum(dim=2) / denom          # (B, C)
        masked_x = x.masked_fill(m == 0, float('-inf'))
        mx = masked_x.max(dim=2).values           # (B, C)
        mx = torch.nan_to_num(mx, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.cat([avg, mx], dim=1)         # (B, 2C)


class StrippedCNN(nn.Module):
    """3-layer 1D CNN on raw per-observation features, no positional encoding.

    Operates on (B, T, in_dim) — transposes to (B, in_dim, T) for Conv1d internally.
    """
    def __init__(self, in_dim=4, hidden=64, tail=32, head_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = MaskedConv1dBlock(in_dim,  hidden, 5, dropout)
        self.conv2 = MaskedConv1dBlock(hidden, hidden, 5, dropout)
        self.conv3 = MaskedConv1dBlock(hidden, tail,    3, dropout)
        self.pool  = MaskedAvgMaxPool()  # concat → (B, 2*tail = 64)
        self.head = nn.Sequential(
            nn.Linear(2 * tail, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x, mask):
        x = x.transpose(1, 2)      # (B, T, C) → (B, C, T)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x, mask)
        return self.head(x).squeeze(-1)  # logits (B,)


# Smoke test
dummy = StrippedCNN(in_dim=N_INPUT, hidden=HIDDEN, tail=TAIL,
                    head_hidden=HEAD_HIDDEN, dropout=DROPOUT)
n_params = sum(p.numel() for p in dummy.parameters() if p.requires_grad)
print(f'StrippedCNN parameters: {n_params:,}')
x0 = torch.randn(4, 50, N_INPUT)
m0 = torch.ones(4, 50, dtype=torch.bool)
with torch.no_grad():
    y0 = dummy(x0, m0)
print(f'Smoke forward: in={tuple(x0.shape)}, mask={tuple(m0.shape)}, out={tuple(y0.shape)})')
del dummy, x0, m0, y0

StrippedCNN parameters: 30,497
Smoke forward: in=(4, 50, 4), mask=(4, 50), out=(4,))


## 5. Dataset & collate function

Variable-length per-star sequences → padded to longest in batch + boolean mask.

In [5]:
class StarDataset(Dataset):
    def __init__(self, sequences, labels):
        self.data = list(zip(sequences, labels))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, lab = self.data[idx]
        return torch.tensor(seq), torch.tensor(lab, dtype=torch.float32)


def collate_fn(batch):
    """Pad variable-length sequences to longest in batch, return (padded, mask, labels)."""
    sequences, labels = zip(*batch)
    lengths = [s.shape[0] for s in sequences]
    max_len = max(lengths)
    padded = torch.zeros(len(sequences), max_len, N_INPUT)
    mask   = torch.zeros(len(sequences), max_len, dtype=torch.bool)
    for i, (seq, length) in enumerate(zip(sequences, lengths)):
        padded[i, :length] = seq
        mask[i, :length] = True
    labels = torch.stack(labels)
    return padded, mask, labels

print('Dataset + collate defined.')

Dataset + collate defined.


## 6. Training function (one fold)

Trains a fresh `StrippedCNN` from scratch on `train_idx` stars for `N_EPOCHS` epochs, then returns predicted probabilities on `test_idx` stars. **Per-fold standardization** (fit on train stars only) and **per-fold truncation** to `MAX_SEQ_LEN` prevent information leakage.

In [6]:
def train_one_fold(train_idx, test_idx, rep_seed, verbose=False):
    """Train a fresh StrippedCNN on train_idx; return probabilities on test_idx."""
    torch.manual_seed(rep_seed); torch.cuda.manual_seed_all(rep_seed)
    np.random.seed(rep_seed)

    # ── Gather raw sequences for train and test stars ──
    train_seqs_raw = [star_seqs[stars[i]] for i in train_idx]
    test_seqs_raw  = [star_seqs[stars[i]] for i in test_idx]
    y_train_fold = labels[train_idx].astype(np.float32).tolist()
    y_test_fold  = labels[test_idx].astype(np.float32).tolist()

    # ── Per-fold standardization: fit on TRAIN observations only ──
    train_all = np.concatenate(train_seqs_raw, axis=0)  # (n_train_obs, 4)
    feat_mean = train_all.mean(axis=0, keepdims=True)
    feat_std  = np.clip(train_all.std(axis=0, keepdims=True), 1e-8, None)

    def standardize(seq_list):
        return [(s.astype(np.float32) - feat_mean) / feat_std for s in seq_list]

    train_seqs = standardize(train_seqs_raw)
    test_seqs  = standardize(test_seqs_raw)

    # ── Truncate to MAX_SEQ_LEN ──
    for seq_list in [train_seqs, test_seqs]:
        for i in range(len(seq_list)):
            if len(seq_list[i]) > MAX_SEQ_LEN:
                seq_list[i] = seq_list[i][:MAX_SEQ_LEN]

    # ── DataLoader ──
    train_ds = StarDataset(train_seqs, y_train_fold)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, drop_last=False)

    # ── Model ──
    model = StrippedCNN(in_dim=N_INPUT, hidden=HIDDEN, tail=TAIL,
                        head_hidden=HEAD_HIDDEN, dropout=DROPOUT).to(device)

    # ── Loss with per-fold pos_weight ──
    n_pos = int(sum(y_train_fold))
    n_neg = int(len(y_train_fold) - n_pos)
    pos_w = torch.tensor([math.sqrt(n_neg / n_pos)]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # ── Cosine LR with warmup ──
    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(1, N_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = LambdaLR(optimizer, lr_lambda)

    # ── Train ──
    model.train()
    for ep in range(N_EPOCHS):
        for padded, mask, yb in train_loader:
            padded, mask, yb = padded.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(padded, mask), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
        scheduler.step()
        if verbose and (ep + 1) % 10 == 0:
            print(f'    ep {ep+1:3d}: loss={loss.item():.4f}')

    # ── Predict on held-out fold ──
    model.eval()
    test_ds = StarDataset(test_seqs, y_test_fold)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             collate_fn=collate_fn, drop_last=False)
    all_logits = []
    with torch.no_grad():
        for padded, mask, _ in test_loader:
            padded, mask = padded.to(device), mask.to(device)
            all_logits.append(model(padded, mask).cpu().numpy())
    logits = np.concatenate(all_logits)

    # Defensive NaN handling (training divergence guard)
    logits = np.nan_to_num(logits, nan=0.0, posinf=35.0, neginf=-35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0).astype(np.float32)
    return probs

print('train_one_fold defined.')

train_one_fold defined.


## 7. OOF Evaluation — 5-fold StratifiedKFold × 5 reps

**Identical protocol to `lightgbm.ipynb` cell 5** so the comparison is honest:

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=rep_seed)`
- Per rep: every star gets a prediction from a CNN that never saw it.
- Per rep: PR-AUC, ROC-AUC, F1, F0.5, precision, recall on OOF predictions.
- Threshold: F1-optimal from the OOF PR curve (held-out, no leakage).
- Combined OOF: average probabilities across all 5 reps.

**25 total CNN fits** (5 folds × 5 reps). Each fit: ~60 epochs × ~1620 train stars. Expect ~30-60s per fold on T4/P100 → ~25-50 minutes total on Kaggle.

In [7]:
all_oof_probs = np.zeros((n_stars, N_REPS), dtype=np.float32)
rep_metrics   = {'rep': [], 'pr_auc': [], 'roc_auc': [],
                 'f1': [], 'f05': [], 'precision': [], 'recall': []}

print(f"{'='*72}")
print(f"OUT-OF-FOLD EVALUATION  (1D CNN on raw RV, no positional encoding)")
print(f"{'='*72}")
print(f"Protocol: {N_FOLDS}-fold StratifiedKFold x {N_REPS} reps, seeds 42-{41+N_REPS}")
print(f"CNN: StrippedCNN({N_INPUT}→{HIDDEN}→{TAIL}), {N_EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}")
print(f"Features: {FEATURE_COLS}")
print(f"Per-rep fits: {N_FOLDS} | Total fits: {N_FOLDS * N_REPS}")
print()

for rep in range(N_REPS):
    oof_probs = np.zeros(n_stars, dtype=np.float32)
    rep_seed  = 42 + rep  # exactly mirrors lightgbm's random_state schedule
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    fold_times = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), labels)):
        t0 = time.time()
        probs = train_one_fold(train_idx, test_idx, rep_seed, verbose=False)
        if not np.all(np.isfinite(probs)):
            n_bad = int((~np.isfinite(probs)).sum())
            print(f'  !!! rep {rep} fold {fold_idx}: {n_bad} non-finite probs replaced with 0.5')
            probs = np.where(np.isfinite(probs), probs, 0.5)
        oof_probs[test_idx] = probs
        fold_times.append(time.time() - t0)
        if (fold_idx + 1) % 2 == 0:
            print(f'  rep {rep} fold {fold_idx+1:2d}/{N_FOLDS} done ({np.mean(fold_times):.0f}s/fold avg)')
        del probs
    all_oof_probs[:, rep] = oof_probs

    # Per-rep metrics (same as lightgbm cell 5)
    roc = roc_auc_score(labels, oof_probs)
    pr  = average_precision_score(labels, oof_probs)
    vp, vr, vt = precision_recall_curve(labels, oof_probs)
    vf1 = 2 * vp * vr / (vp + vr + 1e-8)
    bi = int(np.argmax(vf1))
    thr = float(vt[bi]) if bi < len(vt) else 0.5
    preds = (oof_probs >= thr).astype(int)
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1  = f1_score(labels, preds, zero_division=0)
    f05 = fbeta_score(labels, preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f'  Rep {rep} seed={rep_seed}: PR-AUC={pr:.4f}  ROC={roc:.4f}  F1={f1:.4f}  F0.5={f05:.4f}  P={prc:.3f}  R={rec:.3f}  (thr={thr:.3f})')

print(f"\nOOF evaluation complete.  {N_FOLDS * N_REPS} fits total.")

OUT-OF-FOLD EVALUATION  (1D CNN on raw RV, no positional encoding)
Protocol: 5-fold StratifiedKFold x 5 reps, seeds 42-46
CNN: StrippedCNN(4→64→32), 60 epochs, batch=32, lr=0.001
Features: ['rv_centered', 'rv_err', 'RHKp', 'Halpha']
Per-rep fits: 5 | Total fits: 25

  rep 0 fold  2/5 done (21s/fold avg)
  rep 0 fold  4/5 done (19s/fold avg)
  Rep 0 seed=42: PR-AUC=0.3535  ROC=0.6650  F1=0.4177  F0.5=0.3265  P=0.285  R=0.781  (thr=0.362)
  rep 1 fold  2/5 done (16s/fold avg)
  rep 1 fold  4/5 done (16s/fold avg)
  Rep 1 seed=43: PR-AUC=0.3505  ROC=0.6636  F1=0.4214  F0.5=0.3248  P=0.282  R=0.835  (thr=0.329)
  rep 2 fold  2/5 done (17s/fold avg)
  rep 2 fold  4/5 done (16s/fold avg)
  Rep 2 seed=44: PR-AUC=0.3420  ROC=0.6688  F1=0.4180  F0.5=0.3261  P=0.284  R=0.788  (thr=0.351)
  rep 3 fold  2/5 done (16s/fold avg)
  rep 3 fold  4/5 done (16s/fold avg)
  Rep 3 seed=45: PR-AUC=0.3296  ROC=0.6603  F1=0.4172  F0.5=0.3288  P=0.288  R=0.756  (thr=0.346)
  rep 4 fold  2/5 done (16s/fold avg)

## 8. Summary + head-to-head with LightGBM

Anchor numbers (from `lightgbm.ipynb` cell 5):
- LightGBM (39 V4 features, OOF): PR-AUC = 0.6150, ROC = 0.8461, F1 = 0.5802

If this CNN on raw RV data can approach or exceed those numbers without any hand-engineered aggregate features, it means the raw temporal structure carries signal that summary statistics miss.

In [8]:
rep_df = pd.DataFrame(rep_metrics)

# LightGBM anchors (from lightgbm.ipynb cell 5 OOF summary)
LGB_PR  = 0.6150
LGB_ROC = 0.8461
LGB_F1  = 0.5802

print(f"{'='*72}")
print(f"RAW-RV CNN (4-channel, no pos encoding) — OOF SUMMARY")
print(f"{'='*72}")
print(f"\nPer-rep PR-AUC:")
for _, row in rep_df.iterrows():
    print(f"  Rep {int(row['rep'])}: {row['pr_auc']:.4f}  ROC={row['roc_auc']:.4f}")

print(f"\nAggregate (n={N_REPS} reps):")
for m in ['pr_auc', 'roc_auc', 'f1', 'f05', 'precision', 'recall']:
    v = rep_df[m].values
    print(f"  {m:11s}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}  (min={v.min():.4f}, max={v.max():.4f})")

# Combined OOF (probabilities averaged across reps)
avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(labels, avg_oof)
combined_roc = roc_auc_score(labels, avg_oof)
vp, vr, vt = precision_recall_curve(labels, avg_oof)
vf1 = 2 * vp * vr / (vp + vr + 1e-8)
bi = int(np.argmax(vf1))
thr = float(vt[bi]) if bi < len(vt) else 0.5
combined_preds = (avg_oof >= thr).astype(int)
combined_f1   = f1_score(labels, combined_preds, zero_division=0)
combined_f05  = fbeta_score(labels, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(labels, combined_preds)
tn, fp, fn, tp = cm.ravel()
combined_p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
combined_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"\n{'='*72}")
print(f"COMBINED OOF (probabilities averaged across {N_REPS} reps)")
print(f"{'='*72}")
print(f"  PR-AUC:  {combined_pr:.4f}")
print(f"  ROC-AUC: {combined_roc:.4f}")
print(f"  F1:      {combined_f1:.4f}  F0.5={combined_f05:.4f}  (threshold={thr:.4f})")
print(f"  P={combined_p:.3f}  R={combined_r:.3f}  (TN={tn} FP={fp} FN={fn} TP={tp})")

print(f"\n{'='*72}")
print(f"HEAD-TO-HEAD: Raw-RV CNN vs LightGBM (39 V4 features)")
print(f"{'='*72}")
print(f"                          PR-AUC    ROC-AUC    F1")
print(f"  RAW-RV CNN (rep mean):   {rep_df['pr_auc'].mean():.4f}    {rep_df['roc_auc'].mean():.4f}     {rep_df['f1'].mean():.4f}")
print(f"  RAW-RV CNN (combined):   {combined_pr:.4f}    {combined_roc:.4f}     {combined_f1:.4f}")
print(f"  LightGBM 39 V4 (OOF):    {LGB_PR:.4f}    {LGB_ROC:.4f}     {LGB_F1:.4f}")
delta_pr = combined_pr - LGB_PR
print(f"\n  Delta vs LightGBM: {delta_pr:+.4f}  (positive = CNN beats LightGBM)")

if delta_pr > 0:
    print(f"\n  >>> Raw-RV CNN (+{delta_pr:.4f}) BEATS LightGBM 39 V4 features. Temporal structure carries signal!")
elif delta_pr > -0.03:
    print(f"\n  >>> Raw-RV CNN essentially ties LightGBM (delta < 0.03). Raw sequence ≈ hand-engineered features.")
else:
    print(f"\n  >>> Raw-RV CNN trails LightGBM by > 0.03. Hand-engineered features still dominate.")

RAW-RV CNN (4-channel, no pos encoding) — OOF SUMMARY

Per-rep PR-AUC:
  Rep 0: 0.3535  ROC=0.6650
  Rep 1: 0.3505  ROC=0.6636
  Rep 2: 0.3420  ROC=0.6688
  Rep 3: 0.3296  ROC=0.6603
  Rep 4: 0.3578  ROC=0.6708

Aggregate (n=5 reps):
  pr_auc     : 0.3467 +/- 0.0111  (min=0.3296, max=0.3578)
  roc_auc    : 0.6657 +/- 0.0042  (min=0.6603, max=0.6708)
  f1         : 0.4198 +/- 0.0031  (min=0.4172, max=0.4245)
  f05        : 0.3261 +/- 0.0018  (min=0.3242, max=0.3288)
  precision  : 0.2839 +/- 0.0031  (min=0.2801, max=0.2881)
  recall     : 0.8074 +/- 0.0481  (min=0.7558, max=0.8767)

COMBINED OOF (probabilities averaged across 5 reps)
  PR-AUC:  0.3636
  ROC-AUC: 0.6728
  F1:      0.4213  F0.5=0.3229  (threshold=0.3102)
  P=0.279  R=0.856  (TN=647 FP=949 FN=62 TP=368)

HEAD-TO-HEAD: Raw-RV CNN vs LightGBM (39 V4 features)
                          PR-AUC    ROC-AUC    F1
  RAW-RV CNN (rep mean):   0.3467    0.6657     0.4198
  RAW-RV CNN (combined):   0.3636    0.6728     0.4213
  LightG

## 9. Bootstrap 95% CIs + save results

Uses `split.bootstrap_roc_auc` and `split.bootstrap_pr_auc` (same as lightgbm).

In [10]:
from split import bootstrap_roc_auc, bootstrap_pr_auc

pr_point, pr_lo, pr_hi = bootstrap_pr_auc(labels, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(labels, avg_oof)
print(f"BOOTSTRAP 95% CI (200 resamples):")
print(f"  PR-AUC:  {pr_point:.4f}  [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  ROC-AUC: {roc_point:.4f}  [{roc_lo:.4f}, {roc_hi:.4f}]")

# Save OOF results
out_path = '/kaggle/working/cnn_raw_oof_results.json'
with open(out_path, 'w') as f:
    _json.dump({
        'n_channels': N_INPUT,
        'feature_cols': FEATURE_COLS,
        'n_epochs': N_EPOCHS,
        'n_reps': N_REPS,
        'n_folds': N_FOLDS,
        'max_seq_len': MAX_SEQ_LEN,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'hidden': HIDDEN,
        'tail': TAIL,
        'head_hidden': HEAD_HIDDEN,
        'dropout': DROPOUT,
        'all_oof_probs': all_oof_probs.tolist(),
        'rep_metrics': rep_metrics,
        'labels': labels.tolist(),
        'stars': stars,
        'combined_pr': combined_pr,
        'combined_roc': combined_roc,
        'combined_f1': combined_f1,
        'combined_f05': combined_f05,
        'bootstrap_pr': [pr_point, pr_lo, pr_hi],
        'bootstrap_roc': [roc_point, roc_lo, roc_hi],
    }, f)
print(f"\nSaved OOF results to {out_path}")
print(f"FINAL: Raw-RV CNN combined OOF PR-AUC = {combined_pr:.4f}")

BOOTSTRAP 95% CI (200 resamples):
  PR-AUC:  0.3636  [0.3298, 0.4072]
  ROC-AUC: 0.6728  [0.6493, 0.7009]

Saved OOF results to /kaggle/working/cnn_raw_oof_results.json
FINAL: Raw-RV CNN combined OOF PR-AUC = 0.3636
